# Partie 1 — Data Profiling : US Accidents (2016-2023)

**Objectif :** Analyser la qualité structurelle du dataset US Accidents (~7,7 M de lignes).

| Section | Contenu |
|---------|----------|
| 1 | Vue d'ensemble : nb lignes / nb colonnes |
| 2 | Schéma et types de données |
| 3 | Taux de valeurs manquantes |
| 4 | Détection des doublons |
| 5 | Statistiques descriptives (numériques) |
| 6 | Distribution des variables catégorielles |
| 7 | Détection des outliers (IQR + z-score) |

> **Note technique :** Les calculs de comptage/taux s'effectuent sur Spark (dataset complet).  
> Les graphiques utilisent un échantillon de 5% converti en Pandas.


In [ ]:
# --- Configuration du chemin projet ---
import sys
from pathlib import Path

# Ajout de la racine du projet au PYTHONPATH
RACINE = Path().resolve().parent
if str(RACINE) not in sys.path:
    sys.path.insert(0, str(RACINE))

print(f'Racine projet : {RACINE}')

In [ ]:
# --- Imports ---
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import seaborn as sns

matplotlib.use('Agg')  # Mode fichier (compatible tous OS)
%matplotlib inline

from src.profiling.profiler import (
    creer_session_spark,
    charger_parquet,
    vue_ensemble,
    schema_et_types,
    analyser_valeurs_manquantes,
    detecter_doublons,
    stats_descriptives_numeriques,
    distribution_categorielles,
    detecter_outliers,
    generer_graphiques,
    DIR_FIGURES, DIR_METRICS
)

print('Imports OK')

In [ ]:
# --- Démarrage Spark ---
spark = creer_session_spark('Notebook_01_Profiling')
print(f'Spark {spark.version} démarré')

In [ ]:
# --- Chargement du dataset Parquet ---
df = charger_parquet(spark)
print(f'Dataset chargé : {len(df.columns)} colonnes')

---
## 1. Vue d'ensemble

In [ ]:
res_vue = vue_ensemble(df)
print(f"\n→ {res_vue['nb_lignes']:,} lignes × {res_vue['nb_colonnes']} colonnes")

---
## 2. Schéma et types de données

In [ ]:
df_schema = schema_et_types(df)
# Affichage enrichi dans le notebook
display(df_schema.style.set_caption('Schéma du dataset US Accidents'))

In [ ]:
# Résumé par catégorie de type
print('Répartition par catégorie de type :')
display(df_schema['categorie'].value_counts().reset_index())

---
## 3. Valeurs manquantes

In [ ]:
df_missing = analyser_valeurs_manquantes(df)

In [ ]:
# Affichage du graphique sauvegardé
fig_path = DIR_FIGURES / 'valeurs_manquantes.png'
if fig_path.exists():
    img = mpimg.imread(str(fig_path))
    plt.figure(figsize=(12, 6))
    plt.imshow(img)
    plt.axis('off')
    plt.show()

# Tableau des colonnes avec plus de 5% de manquants
print('\nColonnes avec > 5% de manquants :')
display(df_missing[df_missing['pct_manquants'] > 5].reset_index(drop=True))

---
## 4. Détection des doublons

In [ ]:
res_doublons = detecter_doublons(df)
display(pd.DataFrame([res_doublons]))

---
## 5. Statistiques descriptives — Variables numériques

In [ ]:
df_stats = stats_descriptives_numeriques(df)
display(df_stats.style.format({
    'min': '{:.2f}', 'max': '{:.2f}', 'moyenne': '{:.2f}',
    'mediane': '{:.2f}', 'ecart_type': '{:.2f}',
    'q1': '{:.2f}', 'q3': '{:.2f}', 'iqr': '{:.2f}'
}).set_caption('Statistiques descriptives'))

---
## 6. Distribution des variables catégorielles

In [ ]:
res_cat = distribution_categorielles(df)

# Affichage par colonne
for col_name, data in res_cat.items():
    print(f'\n--- {col_name} (cardinalité : {data["cardinalite"]:,}) ---')
    display(data['top10'])

---
## 7. Détection des valeurs aberrantes (IQR + z-score)

In [ ]:
df_outliers = detecter_outliers(df, df_stats)
display(df_outliers.style.set_caption('Résultats détection outliers'))

---
## 8. Graphiques — Histogrammes et Boxplots

In [ ]:
generer_graphiques(df)

# Affichage dans le notebook
for fig_name in ['histogrammes_numeriques.png', 'boxplots_numeriques.png']:
    fig_path = DIR_FIGURES / fig_name
    if fig_path.exists():
        img = mpimg.imread(str(fig_path))
        plt.figure(figsize=(14, 8))
        plt.imshow(img)
        plt.axis('off')
        plt.title(fig_name.replace('.png', '').replace('_', ' ').title())
        plt.show()

---
## Synthèse du profilage

In [ ]:
print('=' * 60)
print('  SYNTHÈSE — DATA PROFILING US ACCIDENTS')
print('=' * 60)
print(f"Lignes      : {res_vue['nb_lignes']:,}")
print(f"Colonnes    : {res_vue['nb_colonnes']}")
print(f"Doublons ID : {res_doublons['nb_doublons_id']:,} ({res_doublons['pct_doublons_id']:.4f}%)")
print(f"Doublons CK : {res_doublons['nb_doublons_cle_composite']:,} ({res_doublons['pct_doublons_cle_composite']:.4f}%)")
print()
print('Top 5 colonnes les plus incomplètes :')
display(df_missing[df_missing['pct_manquants'] > 0].head(5)[['colonne', 'nb_manquants', 'pct_manquants']])
print()
print('Top 5 colonnes avec le plus d\'outliers (méthode IQR) :')
df_iqr = df_outliers[df_outliers['methode'] == 'IQR'].sort_values('pct_outliers', ascending=False)
display(df_iqr.head(5)[['colonne', 'nb_outliers', 'pct_outliers']])
print('=' * 60)

In [ ]:
# Arrêt propre de Spark
spark.stop()
print('Session Spark arrêtée.')